<a href="https://colab.research.google.com/github/noor-omar/DECI-colabProject/blob/main/EYOUTH_31005041700343_Library.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Name: Noor Omar

ID: EYOUTH_31005041700343

Now, let's load data from `library.db`. We need the `checkouts` table and the `members` table from this database.

In [ ]:
import sqlite3
import pandas as pd

# Connect to the SQLite database
conn = sqlite3.connect('/content/library.db')

# Load the 'checkouts' table into a DataFrame
df_checkouts_db = pd.read_sql_query("SELECT * FROM checkouts;", conn)
print("Checkouts from library.db (df_checkouts_db):")
display(df_checkouts_db.head())

# Load the 'members' table into a DataFrame
df_members_db = pd.read_sql_query("SELECT * FROM members;", conn)
print("\nMembers from library.db (df_members_db):")
display(df_members_db.head())

# Close the database connection
conn.close()

Checkouts from library.db (df_checkouts_db):


,checkout_id,member_id,book_id,checkout_date,return_date
0,9263,1047,517,2024-10-21,2024-11-07
1,9340,1072,513,2025-08-24,2025-09-01
2,9231,1053,523,2024-02-04,2024-02-16
3,9129,1032,513,2025-06-21,2025-06-29
4,9370,1079,511,2025-11-11,2025-12-03



Members from library.db (df_members_db):


,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05
1,1002,Fares,Saleh,9.0,Maadi,Active,None
2,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23
3,1004,Fares,Wahba,7.0,Maadi,inactive,2024-10-09
4,1005,Youssef,Halim,9.0,Maadi,Active,2024-05-05


Next, let's load the book details from `books.json`.

In [ ]:
# Load book details from books.json
df_books_json = pd.read_json('/content/books.json')
print("\nBooks from books.json (df_books_json):")
display(df_books_json.head())


Books from books.json (df_books_json):


,book_id,genre,pages,publication_year,publisher
0,501,Adventure,128,2017.0,Nile Press
1,502,Adventure,109,2018.0,Delta House
2,503,Historical,259,NaN,Nile Press
3,504,Science,319,2009.0,Cairo Young Readers
4,505,Historical,216,2024.0,Oasis Books


Finally, let's load the additional checkouts data from `summer_checkouts.html`. We'll need to inspect the HTML structure to correctly extract the table.

In [ ]:
# Load additional checkouts from summer_checkouts.html
# pandas can read HTML tables directly
df_checkouts_html = pd.read_html('/content/summer_checkouts.html')[0] # Assuming the first table is the one we need
print("\nCheckouts from summer_checkouts.html (df_checkouts_html):")
display(df_checkouts_html.head())


Checkouts from summer_checkouts.html (df_checkouts_html):


,Member ID,Book ID,Checkout Date
0,1026,522,2025-07-11
1,1049,520,2025-07-11
2,1062,525,2025-07-05
3,1065,520,2025-07-07
4,1104,515,2025-07-07


Now, let's combine all the data sources. First, I'll standardize the column names in `df_checkouts_html` and then concatenate it with `df_checkouts_db` to get a unified checkouts list. After that, I'll merge with book details and member details.

In [ ]:
# Standardize column names in df_checkouts_html to match df_checkouts_db
df_checkouts_html_renamed = df_checkouts_html.rename(columns={
    'Member ID': 'member_id',
    'Book ID': 'book_id',
    'Checkout Date': 'checkout_date'
})

# Select relevant columns from df_checkouts_db for concatenation
# Note: df_checkouts_db has 'checkout_id' and 'return_date' which df_checkouts_html_renamed doesn't
# We will align on common columns or create NaNs where data is missing
common_checkout_cols = ['member_id', 'book_id', 'checkout_date']

# Combine the two checkouts dataframes
df_all_checkouts = pd.concat([
    df_checkouts_db[common_checkout_cols],
    df_checkouts_html_renamed[common_checkout_cols]
], ignore_index=True)

print("Combined Checkouts Data (df_all_checkouts):")
display(df_all_checkouts.head())
print(f"Total checkouts after combining: {len(df_all_checkouts)}")


Combined Checkouts Data (df_all_checkouts):


,member_id,book_id,checkout_date
0,1047,517,2024-10-21
1,1072,513,2025-08-24
2,1053,523,2024-02-04
3,1032,513,2025-06-21
4,1079,511,2025-11-11


Total checkouts after combining: 417


Next, let's merge the combined checkouts with the `df_books_json` to add book details.

In [ ]:
# Merge with book details (df_books_json)
df_combined = pd.merge(df_all_checkouts, df_books_json, on='book_id', how='left')

print("Combined Data with Book Details (df_combined):")
display(df_combined.head())


Combined Data with Book Details (df_combined):


,member_id,book_id,checkout_date,genre,pages,publication_year,publisher
0,1047,517,2024-10-21,Mystery,338,2015.0,Delta House
1,1072,513,2025-08-24,Science,294,2021.0,Oasis Books
2,1053,523,2024-02-04,Historical,276,2018.0,Oasis Books
3,1032,513,2025-06-21,Science,294,2021.0,Oasis Books
4,1079,511,2025-11-11,Historical,117,2016.0,Nile Press


Finally, I'll merge the dataset with `df_members_db` to include member details.

In [ ]:
# Merge with member details (df_members_db)
df_final_checkouts = pd.merge(df_combined, df_members_db, on='member_id', how='left')

print("Final Combined Checkouts Dataset (df_final_checkouts):")
display(df_final_checkouts.head())
print(f"Total records in final dataset: {len(df_final_checkouts)}")

Final Combined Checkouts Dataset (df_final_checkouts):


,member_id,book_id,checkout_date,genre,pages,publication_year,publisher,first_name,last_name,grade,neighborhood,membership_status,join_date
0,1047,517,2024-10-21,Mystery,338,2015.0,Delta House,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25
1,1072,513,2025-08-24,Science,294,2021.0,Oasis Books,Seif,Zaki,9.0,Zamalek,Active,2025-10-21
2,1053,523,2024-02-04,Historical,276,2018.0,Oasis Books,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03
3,1032,513,2025-06-21,Science,294,2021.0,Oasis Books,Nada,Zaki,7.0,Nasr City,Active,2025-10-19
4,1079,511,2025-11-11,Historical,117,2016.0,Nile Press,Rana,Osman,8.0,Shubra,Active,2024-10-27


Total records in final dataset: 417


Now, let's calculate how many books each member has borrowed in total from the combined dataset.

In [ ]:
# Calculate total books borrowed per member
df_member_borrow_counts = df_final_checkouts.groupby('member_id').size().reset_index(name='total_books_borrowed')

print("Total books borrowed per member:")
display(df_member_borrow_counts.head())


Total books borrowed per member:


,member_id,total_books_borrowed
0,1001,1
1,1002,3
2,1003,10
3,1005,4
4,1006,1


Finally, I'll save the complete `df_final_checkouts` DataFrame to a CSV file named `task1_combined_data.csv`.

In [ ]:
# Save the combined DataFrame to a CSV file
df_final_checkouts.to_csv('task1_combined_data.csv', index=False)

print("DataFrame saved to 'task1_combined_data.csv'")

DataFrame saved to 'task1_combined_data.csv'
